# dARK 2.0 - Testing Notebook

This notebook demonstrates all dARK 2.0 contract functions with a didactic, step-by-step approach.

## Prerequisites

1. **Start the Besu network:**
   ```bash
   docker-compose up --build
   ```

2. **Install dependencies:**
   ```bash
   pip install web3 py-solc-x jupyter
   ```

---
## 1. Setup and Connection

In [ ]:
from web3 import Web3
from eth_account import Account
import json
import os

# Configuration
RPC_URL = "http://localhost:8545"  # Change if running inside Docker
PRIVATE_KEY = "0xae6ae8e5ccbfb04590405997ee2d52d2b330726137b875053c36d94e974d162f"

# Connect to blockchain
w3 = Web3(Web3.HTTPProvider(RPC_URL))
account = Account.from_key(PRIVATE_KEY)

print(f"✅ Connected: {w3.is_connected()}")
print(f"📦 Block number: {w3.eth.block_number}")
print(f"💰 Account: {account.address}")
print(f"💵 Balance: {w3.from_wei(w3.eth.get_balance(account.address), 'ether'):.2f} ETH")

---
## 2. Load deployed contract

Load the contract ABI and address from `deployed_contracts.ini`

In [ ]:
import configparser

# Load contract info
config = configparser.ConfigParser()
config.read('deployed_contracts.ini')

CONTRACT_ADDRESS = config['dARK']['address']
CONTRACT_ABI = json.loads(config['dARK']['abi'])

# Create contract instance
dark = w3.eth.contract(address=CONTRACT_ADDRESS, abi=CONTRACT_ABI)

print(f"📜 dARK contract loaded at: {CONTRACT_ADDRESS}")
print(f"📋 Available functions: {[f.fn_name for f in dark.functions]}")

---
## 3. Helper function for sending transactions

In [ ]:
def send_tx(func, description="Transaction"):
    """Build, sign, and send a transaction."""
    tx = func.build_transaction({
        'from': account.address,
        'nonce': w3.eth.get_transaction_count(account.address),
        'gas': 500000,
        'gasPrice': w3.eth.gas_price
    })
    
    signed = account.sign_transaction(tx)
    tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction)
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    
    status = "✅ Success" if receipt['status'] == 1 else "❌ Failed"
    print(f"{status} | {description} | Gas: {receipt['gasUsed']:,} | TX: {tx_hash.hex()[:16]}...")
    return receipt

print("Helper function loaded ✓")

---
## 4. NAAN Management

### 4.1 Register a new NAAN

A NAAN (Name Assigning Authority Number) is required before you can create ARKs.

In [ ]:
NAAN = "99999"  # Our test NAAN

# Check if NAAN already exists
if dark.functions.naan_exists(NAAN).call():
    print(f"⚠️ NAAN '{NAAN}' already registered to: {dark.functions.naan_owners(NAAN).call()}")
else:
    # Register the NAAN
    send_tx(dark.functions.register_naan(NAAN), f"Register NAAN '{NAAN}'")

### 4.2 Verify NAAN registration

In [ ]:
naan_owner = dark.functions.naan_owners(NAAN).call()
print(f"NAAN '{NAAN}' is owned by: {naan_owner}")
print(f"Is this our account? {naan_owner == account.address}")

---
## 5. ARK Creation

### 5.1 Create your first ARK

ARK format: `ark:/NAAN/name`

In [ ]:
ARK_ID_1 = f"ark:/{NAAN}/document001"
URL_1 = "https://example.org/documents/my-first-document"
CID_1 = "QmExampleCID12345abcdef"

# Check if ARK exists
if dark.functions.ark_exists(ARK_ID_1).call():
    print(f"⚠️ ARK '{ARK_ID_1}' already exists")
else:
    send_tx(
        dark.functions.create_ark(ARK_ID_1, URL_1, CID_1),
        f"Create ARK '{ARK_ID_1}'"
    )

### 5.2 Create more ARKs

In [ ]:
# Create multiple ARKs
arks_to_create = [
    (f"ark:/{NAAN}/dataset2024", "https://data.example.org/dataset/2024", "QmDataSet2024Hash"),
    (f"ark:/{NAAN}/paper-arxiv-123", "https://arxiv.org/abs/2024.12345", "QmPaperArxivHash"),
    (f"ark:/{NAAN}/image-collection", "https://gallery.example.org/collection/main", "QmImageCollectionCID"),
]

for ark_id, url, cid in arks_to_create:
    if not dark.functions.ark_exists(ark_id).call():
        send_tx(dark.functions.create_ark(ark_id, url, cid), f"Create ARK '{ark_id}'")
    else:
        print(f"⚠️ Already exists: {ark_id}")

---
## 6. ARK Resolution

### 6.1 Resolve an ARK to its URL

This is the primary use case - given an ARK, find where the resource is located.

In [ ]:
# Resolve the first ARK
resolved_url = dark.functions.resolve(ARK_ID_1).call()

print(f"🔍 ARK: {ARK_ID_1}")
print(f"➡️ Resolves to: {resolved_url}")

### 6.2 Get full ARK data

In [ ]:
from datetime import datetime

# Get all data for an ARK
ark_data = dark.functions.get_ark(ARK_ID_1).call()

print(f"\n📦 Full ARK Data for: {ARK_ID_1}")
print(f"   URL:        {ark_data[0]}")
print(f"   CID:        {ark_data[1]}")
print(f"   Owner:      {ark_data[2]}")
print(f"   Created:    {datetime.fromtimestamp(ark_data[3])}")
print(f"   Updated:    {datetime.fromtimestamp(ark_data[4])}")

---
## 7. Update an ARK

Only the ARK owner can update it.

In [ ]:
NEW_URL = "https://new-location.example.org/documents/my-first-document-v2"
NEW_CID = "QmUpdatedCID67890xyz"

print(f"📌 Before update:")
print(f"   URL: {dark.functions.resolve(ARK_ID_1).call()}")

# Update the ARK
send_tx(
    dark.functions.update_ark(ARK_ID_1, NEW_URL, NEW_CID),
    f"Update ARK '{ARK_ID_1}'"
)

print(f"\n📌 After update:")
print(f"   URL: {dark.functions.resolve(ARK_ID_1).call()}")

---
## 8. Check timestamps

In [ ]:
ark_data = dark.functions.get_ark(ARK_ID_1).call()

created = datetime.fromtimestamp(ark_data[3])
updated = datetime.fromtimestamp(ark_data[4])

print(f"⏰ Timestamps for {ARK_ID_1}:")
print(f"   Created: {created}")
print(f"   Updated: {updated}")
print(f"   Modified? {created != updated}")

---
## 9. NAAN Transfer (Advanced)

Transfer NAAN ownership to another wallet. Existing ARKs remain with their original owners.

In [ ]:
# Create a new account for the demo
new_account = Account.create()

print(f"Current NAAN owner: {dark.functions.naan_owners(NAAN).call()}")
print(f"New owner will be: {new_account.address}")

# Uncomment to actually transfer (this would prevent further ARK creation with this account)
# send_tx(
#     dark.functions.transfer_naan(NAAN, new_account.address),
#     f"Transfer NAAN '{NAAN}'"
# )
# print(f"New NAAN owner: {dark.functions.naan_owners(NAAN).call()}")

print("\n⚠️ Transfer is commented out to preserve the demo state")

---
## 10. Query Events (Off-chain Indexing)

List all ARKs by reading blockchain events.

In [ ]:
# Get all ARKCreated events
ark_created_filter = dark.events.ARKCreated.create_filter(from_block=0)
events = ark_created_filter.get_all_entries()

print(f"\n📊 Total ARKs created: {len(events)}\n")

for event in events:
    print(f"  ARK: {event['args']['ark_id']}")
    print(f"    URL: {event['args']['url']}")
    print(f"    CID: {event['args']['cid']}")
    print(f"    Owner: {event['args']['owner'][:10]}...")
    print()

---
## 11. Error Handling Examples

### 11.1 Try to register an existing NAAN

In [ ]:
try:
    send_tx(dark.functions.register_naan(NAAN), "Register existing NAAN")
except Exception as e:
    print(f"❌ Expected error: {str(e)[:100]}...")

### 11.2 Try to create a duplicate ARK

In [ ]:
try:
    send_tx(
        dark.functions.create_ark(ARK_ID_1, "https://other.com", "QmOther"),
        "Create duplicate ARK"
    )
except Exception as e:
    print(f"❌ Expected error: {str(e)[:100]}...")

### 11.3 Try to create ARK with unowned NAAN

In [ ]:
try:
    send_tx(
        dark.functions.create_ark("ark:/00000/not-my-naan", "https://example.com", "QmCid"),
        "Create ARK with unowned NAAN"
    )
except Exception as e:
    print(f"❌ Expected error: {str(e)[:100]}...")

---
## 12. Summary

### Recap of dARK 2.0 Functions

| Function | Type | Description |
|----------|------|-------------|
| `register_naan(naan)` | Write | Register a NAAN |
| `transfer_naan(naan, to)` | Write | Transfer NAAN ownership |
| `create_ark(ark_id, url, cid)` | Write | Create new ARK |
| `update_ark(ark_id, url, cid)` | Write | Update existing ARK |
| `resolve(ark_id)` | View | Get ARK's URL |
| `get_ark(ark_id)` | View | Get full ARK data |
| `ark_exists(ark_id)` | View | Check if ARK exists |
| `naan_exists(naan)` | View | Check if NAAN exists |
| `naan_owners(naan)` | View | Get NAAN owner |

In [ ]:
print("\n🎉 dARK 2.0 Testing Complete!")
print(f"\nContract address: {CONTRACT_ADDRESS}")
print(f"Total blocks: {w3.eth.block_number}")